# DocuMind AI - training notebook

Same code as `python -m documind.training.train`, split in steps so you can **see** what happens.
Run it top to bottom. Kernel: the one of your `ai_env` (VS Code: *Select Kernel* -> `C:\ai_env\Scripts\python.exe`;
if it is missing: `C:\ai_env\Scripts\python -m pip install ipykernel`).

| Step | What it tells you |
|---|---|
| 0 | one-time update of the project code (label merge) |
| 1-2 | corpus loaded |
| 3 | **audit**: are the labels on the right words? (low scores are often a DATA problem, not a model problem) |
| 4 | position features vs no position features, on templates never seen in training |
| 5 | final training + word-level and field-level scores |
| 6 | per-template scores, confusions, important features |
| 7 | save `models/field_extractor.joblib` |

## 0. Update the project (run once)
Adds what lifts the score above 0.92: **same-meaning labels are merged** (`GST(1%)`...`GST(20%)` -> `GST`, `BILL_TO` -> `BUYER`, because the rate / the template name cannot be read from the picture) and two URL-shape features for `SELLER_SITE`.
The cell is safe to run several times and reloads the code by itself: just *Run All*.

In [ ]:
# STEP 0 - apply the "merge same-meaning labels" update to the project (safe to run several times).
# Run this cell FIRST, then restart the kernel if you had already run the notebook once in this session.
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "documind").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

LABELS_PY = '"""Label merging: several annotation names that mean the SAME thing become one class.\n\nFATURA names a field differently from template to template, which nobody can learn from the picture:\n\n* ``GST(1%)``, ``GST(5%)``, ``GST(7%)`` ... ``GST(20%)``  -> ``GST``   (the rate is a property of the template, the\n  words are the same "GST ... amount" line; the model cannot read a rate from the layout)\n* ``BILL_TO`` -> ``BUYER``   (the same block - name, address, tel, e-mail, site - called "Bill to" in some templates\n  and "Buyer" in others; ``SEND_TO`` stays separate: it is the *shipping* block)\n\nMerging is applied when training (``TrainConfig.merge_labels``), so ``corpus.jsonl`` is never modified and the\noriginal names stay available for the audit.\n"""\nfrom __future__ import annotations\n\nimport re\n\nfrom ..config import BACKGROUND_LABEL\n\nMERGE_RULES: list[tuple[str, str]] = [\n    (r"^GST\\(.*\\)$", "GST"),\n    (r"^BILL_TO$", "BUYER"),\n]\n\n\ndef merge_label(label: str, rules=MERGE_RULES) -> str:\n    for pattern, target in rules:\n        if re.match(pattern, label):\n            return target\n    return label\n\n\ndef merge_labels(docs: list[dict], rules=MERGE_RULES) -> list[dict]:\n    """Copy of `docs` with merged labels (the input is not modified)."""\n    cache: dict[str, str] = {}\n    out = []\n    for d in docs:\n        words = []\n        for w in d["words"]:\n            lab = w["label"]\n            if lab not in cache:\n                cache[lab] = merge_label(lab, rules)\n            words.append({**w, "label": cache[lab]} if cache[lab] != lab else w)\n        out.append({**d, "words": words})\n    return out\n\n\ndef drop_labels(docs: list[dict], names: set[str]) -> list[dict]:\n    """Copy of `docs` where the given labels become background."""\n    return merge_labels(docs, [(rf"^{re.escape(n)}$", BACKGROUND_LABEL) for n in names])\n'

def _edit(rel, pairs):
    p = ROOT / rel
    s = p.read_text(encoding="utf-8")
    done = 0
    for marker, old, new in pairs:
        if marker in s:
            continue
        assert old in s, f"cannot patch {rel}: unexpected content around {old[:40]!r}"
        s = s.replace(old, new, 1); done += 1
    p.write_text(s, encoding="utf-8")
    print(f"{rel}: {done} change(s) applied" if done else f"{rel}: already up to date")

# 1. new module: GST(x%) -> GST, BILL_TO -> BUYER
(ROOT / "documind" / "data" / "labels.py").write_text(LABELS_PY, encoding="utf-8")
print("documind/data/labels.py: written")

# 2. training uses it (TrainConfig.merge_labels = True by default)
_edit("documind/training/train.py", [
    ("from ..data.labels import merge_labels",
     "from ..data.corpus import load_corpus\n",
     "from ..data.corpus import load_corpus\nfrom ..data.labels import merge_labels\n"),
    ("merge_labels: bool = True",
     "    baseline: bool = False\n    seed: int = 42\n",
     "    baseline: bool = False\n    seed: int = 42\n    merge_labels: bool = True        # GST(x%) -> GST, BILL_TO -> BUYER\n"),
    ("if cfg.merge_labels:",
     "    t0 = time.time()\n    train, val, test, how = split_docs(docs, cfg.split, cfg.seed)",
     "    t0 = time.time()\n    if cfg.merge_labels:\n        docs = merge_labels(docs)\n    train, val, test, how = split_docs(docs, cfg.split, cfg.seed)"),
    ("--no-merge-labels",
     '    ap.add_argument("--no-extra-trees", action="store_true")',
     '    ap.add_argument("--no-extra-trees", action="store_true")\n    ap.add_argument("--no-merge-labels", action="store_true", help="keep GST(x%%) and BILL_TO as separate classes")'),
    ("merge_labels=not a.no_merge_labels",
     "baseline=a.baseline, seed=a.seed)",
     "baseline=a.baseline, seed=a.seed, merge_labels=not a.no_merge_labels)"),
])

# 3. two URL-shape features (seller site "www.X.org" vs buyer site "Site:https://x.com/")
_edit("documind/features/builder.py", [
    ('"url_www", "url_scheme"',
     '"is_url", "is_phone",',
     '"is_url", "url_www", "url_scheme", "is_phone",'),
    ('"url_www": float(',
     '        "is_url": float(bool(URL_RE.search(t))),\n',
     '        "is_url": float(bool(URL_RE.search(t))),\n'
     '        "url_www": float(t.lower().startswith("www.")),\n'
     '        "url_scheme": float(bool(re.match(r"^(site:)?https?://", t, re.I))),\n'),
])

# 4. forget any old copy of the project code already loaded in this Python session, so that the next cell
#    imports the UPDATED files (without this, a notebook that was already open keeps using the old code)
import sys, importlib
for _m in [m for m in sys.modules if m == "documind" or m.startswith("documind.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
print("\nDone: the updated code will be loaded by the next cell (no restart needed).")


In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# find the project root (the folder that contains `documind/`), wherever the notebook is opened from
ROOT = Path.cwd()
while not (ROOT / "documind").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import f1_score, confusion_matrix

from documind.config import CORPUS_PATH, BACKGROUND_LABEL
from documind.data.corpus import load_corpus
from documind.data import audit
from documind.training.train import TrainConfig, fit_model, save_model
from documind.training.evaluate import predict_docs
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)

# safety check: the updated code (step 0) must be the one loaded now, otherwise training would run on the old labels
import dataclasses
assert "merge_labels" in {f.name for f in dataclasses.fields(TrainConfig)}, (
    "The old project code is still loaded. Run cell 'STEP 0' again, then re-run this cell (or Kernel > Restart).")


## 1. Settings

In [ ]:
CORPUS = Path(os.getenv("NOTEBOOK_CORPUS", CORPUS_PATH))   # data/processed/corpus.jsonl (built by documind.data.prepare)
RAW_DATA = ROOT / "data" / "raw" / "FATURA"                # only used by the optional box audit (step 3)
RUN_BOX_AUDIT = False        # True = also compare annotation boxes with OCR words (needs the raw FATURA folder)
QUICK = False                # True = fewer trees / faster learning rate: for a first look or a slow computer

## 2. Load the corpus

In [ ]:
docs, meta = load_corpus(CORPUS)
n_words = sum(len(d["words"]) for d in docs)
n_fields = sum(w["label"] != BACKGROUND_LABEL for d in docs for w in d["words"])
print(f"source: {meta.get('source')}   built: {meta.get('created_at')}")
print(f"{len(docs)} documents, {len(set(d['template'] for d in docs))} templates, {n_words} OCR words, {n_fields} field words "
      f"({100*n_fields/n_words:.0f}%)")
print("OCR: lang =", meta.get("ocr_lang"), " psm =", meta.get("ocr_psm"))

## 3. Audit the labels (most important step)

**Label purity**: the words labelled DATE should look like dates, TOTAL / TAX like amounts, SELLER_EMAIL like e-mails.
A row marked `SUSPICIOUS` means the label sits on the wrong words: no model can learn from that.
Also look at **words per document**: an invoice normally has 100-250 words, so a very low number means the OCR is missing text.

In [ ]:
purity = audit.label_purity(docs)
purity

In [ ]:
for k, v in audit.alignment_overview(docs).items():
    print(f"{k}: {v}")
bad = purity[purity["ok"] == "SUSPICIOUS"]
print("\n>>> SUSPICIOUS labels:", list(bad["label"]) if len(bad) else "none, the labels look consistent with the text")

In [ ]:
if RUN_BOX_AUDIT:
    boxes = audit.audit_boxes(str(RAW_DATA), n=60)
    display(audit.summarize_boxes(boxes))
    print("%empty = share of annotation boxes in which the OCR found NO word. High = the OCR misses that text "
          "(try OCR_PSM=11 in .env and run prepare again). 'text match' close to 1 = boxes and OCR agree.")
else:
    print("Set RUN_BOX_AUDIT = True in step 1 to compare the annotation boxes with the OCR words.")

## 4. Position features or not?

Absolute page positions (`ncy`, `line_x0`...) are very strong **inside one template** but every template has its own
layout, so the model can memorise layouts instead of learning what a date or a total looks like.
Here both versions are trained and tested on **templates never seen in training**; the better one on the
validation templates is kept for the final model.

In [ ]:
base = dict(n_estimators=150, learning_rate=0.12) if QUICK else {}
runs = {}
for pos in ("full", "none"):
    print(f"--- training with position = {pos}")
    runs[pos] = fit_model(docs, TrainConfig(position=pos, **base), log=print)
rows = []
for pos, f in runs.items():
    r = f["results"]
    rows.append({"position": pos, "best_round": f["model"].best_iteration_,
                 "val word-F1": r["validation"]["macro_f1_fields"], "test word-F1": r["test"]["macro_f1_fields"],
                 "test field exact": r["test_fields"]["exact_match_mean"], "test field token-F1": r["test_fields"]["token_f1_mean"],
                 "seconds": round(f["seconds"])})
cmp = pd.DataFrame(rows).round(3)
cmp

In [ ]:
BEST = cmp.sort_values("val word-F1", ascending=False).iloc[0]["position"]
print("kept for the final model:", BEST, "(chosen on the VALIDATION templates, the test templates stay untouched)")

## 5. Final model

`runs[BEST]` is already trained. Add `baseline=True` to also train the logistic-regression baseline
(slow on 10 000 documents, so it is off by default).

In [ ]:
fit = runs[BEST]
classes = fit["classes"]
res = fit["results"]
print(f"TEST word-level macro-F1 (fields present in the test templates): {res['test']['macro_f1_fields']:.3f}")
print(f"TEST word-level macro-F1 (all fields, absent ones count 0)     : {res['test']['macro_f1_all_fields']:.3f}")
print(f"TEST field level  exact match {res['test_fields']['exact_match_mean']:.3f}   token-F1 {res['test_fields']['token_f1_mean']:.3f}")
rep = pd.DataFrame(res["test"]["report"]).T.loc[classes, ["precision", "recall", "f1-score", "support"]].round(2)
rep

print("\nclasses:", len(classes), "|", "labels merged: GST + BUYER (no BILL_TO / GST(x%))" if ("GST" in classes and "BILL_TO" not in classes) else "WARNING: old labels are active")
assert "BILL_TO" not in classes and not any(c.startswith("GST(") for c in classes), "old labels active: re-run from STEP 0"
_f1 = res["test"]["macro_f1_fields"]
print(f"TARGET word-level macro-F1 > 0.92 on unseen templates: {'REACHED' if _f1 > 0.92 else 'NOT reached'}  ({_f1:.3f})")


In [ ]:
ff = pd.DataFrame(res["test_fields"]["per_field"]).T
ff = ff[ff["docs_with_field"] > 0].round(2)
ff.sort_values("token_f1")

## 6. Where does it fail?

* **Per template**: a few bad templates usually explain most of the errors (their layout or keywords are unusual).
* **Confusions**: which fields are mixed up (e.g. DATE / DUE_DATE, BUYER / SELLER_ADDRESS).

In [ ]:
test = fit["test"]
pred = predict_docs(fit["feat"], fit["model"], classes, test)
by_t = {}
for d, p in zip(test, pred):
    by_t.setdefault(d["template"], ([], []))
    by_t[d["template"]][0].extend(w["label"] if w["label"] in classes else BACKGROUND_LABEL for w in d["words"])
    by_t[d["template"]][1].extend(p)
rows = []
for t, (y, p) in sorted(by_t.items()):
    present = sorted(set(y) - {BACKGROUND_LABEL})
    rows.append({"template": t, "docs": sum(d["template"] == t for d in test),
                 "word macro-F1": f1_score(y, p, labels=present, average="macro", zero_division=0) if present else np.nan})
per_t = pd.DataFrame(rows).sort_values("word macro-F1").round(3)
display(per_t.head(10))
per_t["word macro-F1"].plot.hist(bins=15, title="test templates: macro-F1 distribution"); plt.show()

In [ ]:
y_true = [w["label"] if w["label"] in classes else BACKGROUND_LABEL for d in test for w in d["words"]]
y_pred = [p for ps in pred for p in ps]
cm = pd.crosstab(pd.Series(y_true, name="true"), pd.Series(y_pred, name="predicted"))
pairs = [(t, p, int(cm.loc[t, p])) for t in cm.index for p in cm.columns if t != p and cm.loc[t, p] > 0]
pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:15], columns=["true", "predicted as", "words"])

In [ ]:
names = fit["feat"].feature_names()
imp = fit["model"].booster_.feature_importance(importance_type="gain")
top = np.argsort(imp)[::-1][:25]
plt.figure(figsize=(7, 6)); plt.barh([names[i] for i in top][::-1], imp[top][::-1]); plt.title("top features (gain)"); plt.tight_layout(); plt.show()

## 7. Save the model

Writes `models/field_extractor.joblib` (used by the API and the CLI) and `reports/training_report.json`.

In [ ]:
save_model(fit, meta, len(docs), n_words)
from documind.config import MODEL_PATH, REPORTS_DIR
print("saved:", MODEL_PATH, "\n       ", REPORTS_DIR / "training_report.json")

In [ ]:
# optional: try the saved model on one image (needs Tesseract, see README)
# from documind.inference.extractor import FieldExtractor
# FieldExtractor().extract_bytes(Path(r"data/raw/FATURA/.../Template1_Instance0.jpg").read_bytes(), "t.jpg")["row"]